Data Collection

#Data Collection von 12.01.2025 und  23.02.2025 für r/politik

In [1]:
import requests
import time
from datetime import datetime, timezone
from dateutil.relativedelta import relativedelta
import pandas as pd
from tqdm.notebook import tqdm
import asyncio
import aiohttp
from aiohttp import ClientResponseError
from pathlib import Path # Added import

# ── Konfiguration (Zeitraum: 12.01.2025 - 23.02.2025) ━━━━━━━━━
DATE_START = datetime(2025, 1, 12, tzinfo=timezone.utc)
DATE_END   = datetime(2025, 2, 23, tzinfo=timezone.utc)
MIN_WORDS  = 3
LIMIT_PER_MONTH = 1000

SUBREDDIT = 'politik'

AS_BASE    = 'https://arctic-shift.photon-reddit.com'
AS_HEADERS = {'User-Agent': 'politik-corpus-collector/1.0 (academic research)'}
AS_LIMIT   = 100

# Konfiguration für Retries
MAX_RETRIES = 5
INITIAL_BACKOFF_DELAY = 1.0  # Sekunden

# ── Helpers ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def _parse_post(post: dict, subreddit: str) -> dict | None:
    # Nur Self-Posts (Textbeiträge) berücksichtigen, da API-Parameter dafür nicht unterstützt wird
    if not post.get('is_self', False):
        return None

    body  = post.get('selftext', '').strip()
    if body in ('[deleted]', '[removed]'):
        body = ''
    title = post.get('title', '').strip()
    text  = (title + ' ' + body).strip()

    if len(text.split()) < MIN_WORDS:
        return None

    ts = post.get('created_utc', 0)
    return {
        'id':           post.get('id'),
        'subreddit':    subreddit,
        'title':        title,
        'body':         body,
        'text':         text,
        'word_count':   len(text.split()),
        'score':        post.get('score', 0),
        'num_comments': post.get('num_comments', 0),
        'created_utc':  datetime.fromtimestamp(int(ts), tz=timezone.utc).strftime('%Y-%m-%d') if ts else None,
        'flair':        post.get('link_flair_text'),
    }

async def _fetch_window(session: aiohttp.ClientSession, subreddit: str, after_ts: int, before_ts: int, monthly_cap: int) -> tuple[str, list]:
    posts_out  = []
    current_before = before_ts
    retry_count = 0
    current_delay = INITIAL_BACKOFF_DELAY

    while len(posts_out) < monthly_cap:
        params = {
            'subreddit': subreddit,
            'after':     after_ts,
            'before':    current_before,
            'limit':     AS_LIMIT,
            'sort':      'desc',
            'sort_type': 'created_utc'
        }

        try:
            async with session.get(f'{AS_BASE}/api/posts/search', params=params, headers=AS_HEADERS, timeout=30) as response:
                response.raise_for_status()
                json_data = await response.json()
                batch = json_data.get('data', [])
            retry_count = 0
            current_delay = INITIAL_BACKOFF_DELAY

        except ClientResponseError as e:
            if e.status == 429:
                if retry_count < MAX_RETRIES:
                    print(f'      [!] Rate limit für {subreddit} ab {datetime.fromtimestamp(after_ts, tz=timezone.utc).strftime("%Y-%m-%d")}. Retry in {current_delay:.1f}s (Versuch {retry_count + 1}/{MAX_RETRIES})')
                    await asyncio.sleep(current_delay)
                    current_delay *= 2
                    retry_count += 1
                    continue
                else:
                    print(f'      [!] Max. Retries erreicht für {subreddit} ab {datetime.fromtimestamp(after_ts, tz=timezone.utc).strftime("%Y-%m-%d")} (429).')
                    break
            else:
                print(f'      [!] API-Fehler für {subreddit} ab {datetime.fromtimestamp(after_ts, tz=timezone.utc).strftime("%Y-%m-%d")}: {e}')
                break
        except asyncio.TimeoutError:
            print(f'      [!] API-Timeout für {subreddit} ab {datetime.fromtimestamp(after_ts, tz=timezone.utc).strftime("%Y-%m-%d")}')
            break
        except Exception as e:
            print(f'      [!] Unerwarteter Fehler für {subreddit} ab {datetime.fromtimestamp(after_ts, tz=timezone.utc).strftime("%Y-%m-%d")}: {e}')
            break

        if not batch: break

        posts_out.extend(batch)

        if len(batch) < AS_LIMIT: break

        oldest_received = min(int(p.get('created_utc', current_before)) for p in batch)
        if oldest_received <= after_ts or oldest_received == current_before:
            break
        current_before = oldest_received
        await asyncio.sleep(0.05)

    return subreddit, posts_out

# ── Haupt-Collector ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

async def collect_arctic_shift() -> pd.DataFrame:
    records = []
    seen    = set()

    fetch_params = []
    current = DATE_START
    while current < DATE_END:
        window_end = min(current + relativedelta(months=1), DATE_END)
        after_ts   = int(current.timestamp())
        before_ts  = int(window_end.timestamp())
        fetch_params.append((SUBREDDIT, after_ts, before_ts, LIMIT_PER_MONTH))
        current = window_end

    connector = aiohttp.TCPConnector(limit=2)
    async with aiohttp.ClientSession(connector=connector) as session:
        tasks = [_fetch_window(session, *params) for params in fetch_params]

        pbar = tqdm(asyncio.as_completed(tasks), total=len(tasks), desc="Total Progress")

        stats = {'n_passed': 0, 'n_filtered': 0}

        for future in pbar:
            subreddit_name, batch_posts = await future
            for post in batch_posts:
                pid = post.get('id')
                if not pid or pid in seen: continue
                seen.add(pid)

                record = _parse_post(post, subreddit_name)
                if record:
                    records.append(record)
                    stats['n_passed'] += 1
                else:
                    stats['n_filtered'] += 1
            pbar.set_postfix(
                passed=f"{stats['n_passed']:,}",
                filtered=f"{stats['n_filtered']:,}",
                refresh=False
            )

    print(f'\n── r/{SUBREDDIT}')
    print(f"  [Fertig] gesamt bestanden: {stats['n_passed']} | gefiltert (kurz/kein Text): {stats['n_filtered']}")

    return pd.DataFrame(records).drop_duplicates('id')

# ── Collection starten ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

df_raw = await collect_arctic_shift()

# In Drive speichern
OUTPUT_DIR = Path('.') # Added definition for OUTPUT_DIR
raw_path = OUTPUT_DIR / 'reddit_politik_raw.csv'
df_raw.to_csv(raw_path, index=False)

print(f"\n{len(df_raw):,} Beiträge gespeichert unter {raw_path}")
display(df_raw.head())

Total Progress:   0%|          | 0/2 [00:00<?, ?it/s]


── r/politik
  [Fertig] gesamt bestanden: 398 | gefiltert (kurz/kein Text): 15

398 Beiträge gespeichert unter reddit_politik_raw.csv


,id,subreddit,title,body,text,word_count,score,num_comments,created_utc,flair
0,1ivvd1r,politik,Kriegsende in Sicht?!,,Kriegsende in Sicht?!,3,1,1,2025-02-22,sonstige
1,1ivtfrl,politik,Frage an die Grünen de Benutzer hier,Ich weiß nicht ob die userbase hier ähnlich gr...,Frage an die Grünen de Benutzer hier Ich weiß ...,342,1,13,2025-02-22,sonstige
2,1ivswq5,politik,"Ich überlege, die CDU zu wählen – suche aber f...",,"Ich überlege, die CDU zu wählen – suche aber f...",11,1,1,2025-02-22,Frage
3,1ivsui2,politik,CDU & FDP Minderheitsregierung,Denkt ihr eine Minderheitsregierung von schwar...,CDU & FDP Minderheitsregierung Denkt ihr eine ...,61,0,3,2025-02-22,Meinung
4,1ivrppk,politik,Getroffene Hunde bellen,Würde gerade auf den Wahlkampfauftritt von Mer...,Getroffene Hunde bellen Würde gerade auf den W...,360,7,20,2025-02-22,Meinung


In [2]:
df_raw.head()

,id,subreddit,title,body,text,word_count,score,num_comments,created_utc,flair
0,1ivvd1r,politik,Kriegsende in Sicht?!,,Kriegsende in Sicht?!,3,1,1,2025-02-22,sonstige
1,1ivtfrl,politik,Frage an die Grünen de Benutzer hier,Ich weiß nicht ob die userbase hier ähnlich gr...,Frage an die Grünen de Benutzer hier Ich weiß ...,342,1,13,2025-02-22,sonstige
2,1ivswq5,politik,"Ich überlege, die CDU zu wählen – suche aber f...",,"Ich überlege, die CDU zu wählen – suche aber f...",11,1,1,2025-02-22,Frage
3,1ivsui2,politik,CDU & FDP Minderheitsregierung,Denkt ihr eine Minderheitsregierung von schwar...,CDU & FDP Minderheitsregierung Denkt ihr eine ...,61,0,3,2025-02-22,Meinung
4,1ivrppk,politik,Getroffene Hunde bellen,Würde gerade auf den Wahlkampfauftritt von Mer...,Getroffene Hunde bellen Würde gerade auf den W...,360,7,20,2025-02-22,Meinung


In [3]:
df_raw.columns

Index(['id', 'subreddit', 'title', 'body', 'text', 'word_count', 'score',
       'num_comments', 'created_utc', 'flair'],
      dtype='object')

#Data Collection von 12.01.2025 und 23.02.2025 für r/politikbrd

In [4]:
import requests
import time
from datetime import datetime, timezone
from dateutil.relativedelta import relativedelta
import pandas as pd
from tqdm.notebook import tqdm
import asyncio
import aiohttp
from aiohttp import ClientResponseError

# ── Konfiguration (Zeitraum: 12.01.2025 - 23.02.2025) ━━━━━━━━━
DATE_START = datetime(2025, 1, 12, tzinfo=timezone.utc)
DATE_END   = datetime(2025, 2, 23, tzinfo=timezone.utc)
MIN_WORDS  = 3
LIMIT_PER_MONTH = 1000

SUBREDDIT = 'PolitikBRD'

AS_BASE    = 'https://arctic-shift.photon-reddit.com'
AS_HEADERS = {'User-Agent': 'politikbrd-corpus-collector/1.0 (academic research)'}
AS_LIMIT   = 100

# Konfiguration für Retries
MAX_RETRIES = 5
INITIAL_BACKOFF_DELAY = 1.0  # Sekunden

# ── Helpers ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def _parse_post(post: dict, subreddit: str) -> dict | None:
    # Nur Self-Posts (Textbeiträge) berücksichtigen, da API-Parameter dafür nicht unterstützt wird
    if not post.get('is_self', False):
        return None

    body  = post.get('selftext', '').strip()
    if body in ('[deleted]', '[removed]'):
        body = ''
    title = post.get('title', '').strip()
    text  = (title + ' ' + body).strip()

    if len(text.split()) < MIN_WORDS:
        return None

    ts = post.get('created_utc', 0)
    return {
        'id':           post.get('id'),
        'subreddit':    subreddit,
        'title':        title,
        'body':         body,
        'text':         text,
        'word_count':   len(text.split()),
        'score':        post.get('score', 0),
        'num_comments': post.get('num_comments', 0),
        'created_utc':  datetime.fromtimestamp(int(ts), tz=timezone.utc).strftime('%Y-%m-%d') if ts else None,
        'flair':        post.get('link_flair_text'),
    }

async def _fetch_window(session: aiohttp.ClientSession, subreddit: str, after_ts: int, before_ts: int, monthly_cap: int) -> tuple[str, list]:
    posts_out  = []
    current_before = before_ts
    retry_count = 0
    current_delay = INITIAL_BACKOFF_DELAY

    while len(posts_out) < monthly_cap:
        params = {
            'subreddit': subreddit,
            'after':     after_ts,
            'before':    current_before,
            'limit':     AS_LIMIT,
            'sort':      'desc',
            'sort_type': 'created_utc'
        }

        try:
            async with session.get(f'{AS_BASE}/api/posts/search', params=params, headers=AS_HEADERS, timeout=30) as response:
                response.raise_for_status()
                json_data = await response.json()
                batch = json_data.get('data', [])
            retry_count = 0
            current_delay = INITIAL_BACKOFF_DELAY

        except ClientResponseError as e:
            if e.status == 429:
                if retry_count < MAX_RETRIES:
                    print(f'      [!] Rate limit für {subreddit} ab {datetime.fromtimestamp(after_ts, tz=timezone.utc).strftime("%Y-%m-%d")}. Retry in {current_delay:.1f}s (Versuch {retry_count + 1}/{MAX_RETRIES})')
                    await asyncio.sleep(current_delay)
                    current_delay *= 2
                    retry_count += 1
                    continue
                else:
                    print(f'      [!] Max. Retries erreicht für {subreddit} ab {datetime.fromtimestamp(after_ts, tz=timezone.utc).strftime("%Y-%m-%d")} (429).')
                    break
            else:
                print(f'      [!] API-Fehler für {subreddit} ab {datetime.fromtimestamp(after_ts, tz=timezone.utc).strftime("%Y-%m-%d")}: {e}')
                break
        except asyncio.TimeoutError:
            print(f'      [!] API-Timeout für {subreddit} ab {datetime.fromtimestamp(after_ts, tz=timezone.utc).strftime("%Y-%m-%d")}')
            break
        except Exception as e:
            print(f'      [!] Unerwarteter Fehler für {subreddit} ab {datetime.fromtimestamp(after_ts, tz=timezone.utc).strftime("%Y-%m-%d")}: {e}')
            break

        if not batch: break

        posts_out.extend(batch)

        if len(batch) < AS_LIMIT: break

        oldest_received = min(int(p.get('created_utc', current_before)) for p in batch)
        if oldest_received <= after_ts or oldest_received == current_before:
            break
        current_before = oldest_received
        await asyncio.sleep(0.05)

    return subreddit, posts_out

# ── Haupt-Collector ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

async def collect_arctic_shift() -> pd.DataFrame:
    records = []
    seen    = set()

    fetch_params = []
    current = DATE_START
    while current < DATE_END:
        window_end = min(current + relativedelta(months=1), DATE_END)
        after_ts   = int(current.timestamp())
        before_ts  = int(window_end.timestamp())
        fetch_params.append((SUBREDDIT, after_ts, before_ts, LIMIT_PER_MONTH))
        current = window_end

    connector = aiohttp.TCPConnector(limit=2)
    async with aiohttp.ClientSession(connector=connector) as session:
        tasks = [_fetch_window(session, *params) for params in fetch_params]

        pbar = tqdm(asyncio.as_completed(tasks), total=len(tasks), desc="Total Progress")

        stats = {'n_passed': 0, 'n_filtered': 0}

        for future in pbar:
            subreddit_name, batch_posts = await future
            for post in batch_posts:
                pid = post.get('id')
                if not pid or pid in seen: continue
                seen.add(pid)

                record = _parse_post(post, subreddit_name)
                if record:
                    records.append(record)
                    stats['n_passed'] += 1
                else:
                    stats['n_filtered'] += 1
            pbar.set_postfix(
                passed=f"{stats['n_passed']:,}",
                filtered=f"{stats['n_filtered']:,}",
                refresh=False
            )

    print(f'\n── r/{SUBREDDIT}')
    print(f"  [Fertig] gesamt bestanden: {stats['n_passed']} | gefiltert (kurz/kein Text): {stats['n_filtered']}")

    return pd.DataFrame(records).drop_duplicates('id')

# ── Collection starten ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

df_raw = await collect_arctic_shift()

# In Drive speichern
raw_path = OUTPUT_DIR / 'reddit_politikbrd_raw.csv'
df_raw.to_csv(raw_path, index=False)

print(f"\n{len(df_raw):,} Beiträge gespeichert unter {raw_path}")
display(df_raw.head())


Total Progress:   0%|          | 0/2 [00:00<?, ?it/s]


── r/PolitikBRD
  [Fertig] gesamt bestanden: 133 | gefiltert (kurz/kein Text): 1345

133 Beiträge gespeichert unter reddit_politikbrd_raw.csv


,id,subreddit,title,body,text,word_count,score,num_comments,created_utc,flair
0,1ivu48e,PolitikBRD,Speed dating Sat 1,Ich habe gerade die Sendung auf Sat 1 gesehen....,Speed dating Sat 1 Ich habe gerade die Sendung...,59,1,1,2025-02-22,None
1,1ivtstc,PolitikBRD,"Ich überlege, die CDU zu wählen – suche aber f...","Hallo zusammen,\n\nich beschäftige mich gerade...","Ich überlege, die CDU zu wählen – suche aber f...",97,0,33,2025-02-22,None
2,1ivf5eh,PolitikBRD,Ist Söder ein Schlüsselpolitiker?,"Gebt eure Meinungen ab, was passiert im Bundes...",Ist Söder ein Schlüsselpolitiker? Gebt eure Me...,58,2,24,2025-02-22,None
3,1ivewaj,PolitikBRD,Community lässt keine Meinungsfreiheit zu,,Community lässt keine Meinungsfreiheit zu,5,0,1,2025-02-22,None
4,1ivelpo,PolitikBRD,Integration von Fachkräften,,Integration von Fachkräften,3,1,2,2025-02-22,None
